In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# تهيئة الكيوبتات

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
عند تنفيذ دائرة على وحدة معالجة كمية (QPU) من IBM&reg;، يُدرج النظام عادةً إعادة تعيين ضمنية في بداية الدائرة للتأكد من تهيئة الكيوبتات إلى الصفر. يتحكم في هذا السلوك العلَم `init_qubits`، الذي يُعيَّن كـ[خيار تنفيذ primitive](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2).

غير أن عدم كمال عملية إعادة التعيين قد يُدخل أخطاءً في تحضير الحالة. للتخفيف من هذا الخطأ، تُدرج وحدة QPU أيضًا وقت تأخير التكرار (أو `rep_delay`) بين الدوائر. لكل backend قيمة افتراضية مختلفة لـ`rep_delay`، لكنها في الغالب تُضبط لتحقيق توازن بين دقة إعادة التعيين وإجمالي وقت التنفيذ. شغِّل `backend.default_rep_delay` للاطلاع على قيمة `rep_delay` الافتراضية لوحدة QPU معينة.

نظرًا لاعتماد جميع وحدات IBM الكمية على تنفيذ معدل التكرار الديناميكي، يمكنك تغيير `rep_delay` لكل مهمة. تُجمَّع الدوائر التي ترسلها في مهمة primitive معًا للتنفيذ على وحدة QPU. تُنفَّذ هذه الدوائر بالتكرار عليها لكل لقطة (shot) مطلوبة؛ يسير التنفيذ عموديًا عبر مصفوفة من الدوائر واللقطات، كما هو موضح في الشكل التالي.

![العمود الأول يمثل اللقطة 0. تُنفَّذ الدوائر بالترتيب من 0 إلى 3. العمود الثاني يمثل اللقطة 1. تُنفَّذ الدوائر بالترتيب من 0 إلى 3. تتبع الأعمدة المتبقية نفس النمط. ](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "مصفوفة التنفيذ العمودي")

نظرًا لإدراج `rep_delay` بين الدوائر، تواجه كل لقطة هذا التأخير. لذلك، كلما خفضت `rep_delay`، انخفض إجمالي وقت تنفيذ وحدة QPU، على حساب ارتفاع معدل خطأ تحضير الحالة، كما توضح الصورة التالية:

![تُظهر هذه الصورة أنه كلما انخفضت قيمة `rep_delay`، ارتفع معدل خطأ تحضير الحالة.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "تأخير التكرار مقابل معدل الخطأ")

إذا ضبطت كلًا من `rep_delay=0` و`init_qubits=False`، تتحقق "دمج" الدوائر، إذ ستبدأ الكيوبتات من الحالة النهائية للقطة السابقة.

تجدر الإشارة إلى أنه رغم تجميع دوائر مهمة primitive معًا لتنفيذها على وحدة QPU، لا يوجد ضمان بشأن ترتيب تنفيذ الدوائر القادمة من وحدات PUB. فعلى سبيل المثال، إذا أرسلت `pubs=[pub1, pub2]`، قد لا تُنفَّذ دوائر `pub1` قبل `pub2`. كما لا يوجد ضمان بأن دوائر المهمة ذاتها ستعمل كدفعة واحدة على وحدة QPU.

## تحديد `rep_delay` لمهمة primitive
> **Note:** تحقق دائمًا من نطاق `rep_delay` المدعوم للوحدة القياسية QPU التي تستخدمها. لا تكون هذه القيم متماثلة لكل QPU وقد تتغير أيضًا بمرور الوقت.
> 
> تجدر الإشارة إلى أن زيادة `rep_delay` سيكون لها تأثير مباشر على وقت التنفيذ واستهلاك الطاقة.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## الخطوات التالية
> **Tip:** - جرِّب مثالًا في درس [خوارزمية التحسين الكمي التقريبي (QAOA)](/tutorials/quantum-approximate-optimization-algorithm).
>   - راجع كيفية [البدء باستخدام Estimator](/guides/get-started-with-estimator).
>   - راجع كيفية [البدء باستخدام Sampler](/guides/get-started-with-sampler).